In [ ]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
    import pandas as pd
    import glob
    import os
    from datetime import datetime, timedelta
    
    # === 1️⃣ Load all .asc files (top 500 rows if available) ===
    data_path = "/kaggle/input/masterclass"
    data = {}
    
    for path in glob.glob(os.path.join(data_path, "*.asc")):
        name = os.path.basename(path).split(".")[0]
    
        # Read CSV with quotechar and dtype=str
        df = pd.read_csv(path, sep=';', quotechar='"',  dtype=str, low_memory=False)
        df.columns = df.columns.str.strip()  # remove whitespace
        data[name] = df
        print(f"{name}: loaded {len(df)} rows")
    


In [ ]:
for i in data :
    print(i," : " ,data[i].columns)

In [ ]:
data["order"].head(5)

In [ ]:
data["order"]["k_symbol"].value_counts()

In [ ]:
import pandas as pd
import numpy as np
import glob
import os

# === Load data ===
data_path = "/kaggle/input/masterclass"
data = {}
for path in glob.glob(os.path.join(data_path, "*.asc")):
    name = os.path.basename(path).split(".")[0]
    df = pd.read_csv(path, sep=';', quotechar='"', dtype=str, low_memory=False)
    df.columns = df.columns.str.strip()
    data[name] = df

# --- Convert numeric/date fields where needed ---
def to_numeric(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')
    return df

data['trans'] = to_numeric(data['trans'], ['amount', 'balance'])
data['loan'] = to_numeric(data['loan'], ['amount', 'duration', 'payments'])
data['order'] = to_numeric(data['order'], ['amount'])
data['card']['issued'] = pd.to_datetime(data['card']['issued'], errors='coerce', format='%y%m%d %H:%M:%S')
data['account']['date'] = pd.to_datetime(data['account']['date'], errors='coerce', format='%y%m%d')
data['trans']['date'] = pd.to_datetime(data['trans']['date'], errors='coerce', format='%y%m%d')

# === 1️⃣ Pick a single client (for example client_id = '12850') ===
client_id = '12850'
client = data['client'][data['client']['client_id'] == client_id]

# --- Join with district info ---
client = client.merge(data['district'], left_on='district_id', right_on='A1', how='left')

# --- Find accounts via disp (OWNER only) ---
disp = data['disp'][ (data['disp']['client_id'] == client_id) & (data['disp']['type'] == 'OWNER') ]
account_ids = disp['account_id'].unique()

# --- Filter related tables for this client’s accounts ---
accounts = data['account'][data['account']['account_id'].isin(account_ids)]
loans = data['loan'][data['loan']['account_id'].isin(account_ids)]
cards = data['card'][data['card']['disp_id'].isin(disp['disp_id'])]
orders = data['order'][data['order']['account_id'].isin(account_ids)]
trans = data['trans'][data['trans']['account_id'].isin(account_ids)]

# === 2️⃣ Aggregate Transactions ===
if len(trans) > 0:
    trans_summary = pd.DataFrame({
        'account_id': trans['account_id'].unique()
    })

    trans_summary['total_transactions'] = trans.groupby('account_id')['trans_id'].count().values
    trans_summary['num_incoming'] = trans.groupby('account_id').apply(lambda x: (x['type'] == 'PRIJEM').sum()).values
    trans_summary['num_outgoing'] = trans.groupby('account_id').apply(lambda x: (x['type'] == 'VYDAJ').sum()).values
    trans_summary['total_incoming'] = trans.groupby('account_id').apply(lambda x: x.loc[x['type'] == 'PRIJEM', 'amount'].sum()).values
    trans_summary['total_outgoing'] = trans.groupby('account_id').apply(lambda x: x.loc[x['type'] == 'VYDAJ', 'amount'].sum()).values
    trans_summary['avg_incoming'] = trans.groupby('account_id').apply(lambda x: x.loc[x['type'] == 'PRIJEM', 'amount'].mean()).values
    trans_summary['avg_outgoing'] = trans.groupby('account_id').apply(lambda x: x.loc[x['type'] == 'VYDAJ', 'amount'].mean()).values
    trans_summary['balance_min'] = trans.groupby('account_id')['balance'].min().values
    trans_summary['balance_max'] = trans.groupby('account_id')['balance'].max().values
    trans_summary['balance_mean'] = trans.groupby('account_id')['balance'].mean().values
    trans_summary['unique_k_symbols'] = trans.groupby('account_id')['k_symbol'].nunique().values
    trans_summary['transaction_span_days'] = trans.groupby('account_id')['date'].apply(lambda x: (x.max() - x.min()).days).values
else:
    trans_summary = pd.DataFrame()

# === 3️⃣ Aggregate Loans, Cards, Orders ===
loan_summary = loans.groupby('account_id').agg({
    'amount': 'sum',
    'duration': 'mean',
    'payments': 'mean',
    'status': lambda x: ','.join(x.unique())
}).rename(columns={'amount': 'loan_amount_total', 'duration': 'loan_avg_duration',
                   'payments': 'loan_avg_payment', 'status': 'loan_status'})

card_summary = cards.groupby('disp_id').agg({
    'type': 'count',
    'issued': 'min'
}).rename(columns={'type': 'num_cards', 'issued': 'earliest_card_issue'})

order_summary = orders.groupby('account_id').agg({
    'amount': ['count', 'mean', 'sum']
})
order_summary.columns = ['num_orders', 'avg_order_amount', 'total_order_amount']

# === 4️⃣ Merge all summaries per account ===
account_summary = accounts.merge(trans_summary, on='account_id', how='left') \
                          .merge(loan_summary, on='account_id', how='left') \
                          .merge(order_summary, on='account_id', how='left')

# === 5️⃣ Attach this back to the client ===
client_full = client.copy()
client_full = client_full.assign(
    num_accounts=len(account_summary),
    total_balance=account_summary['balance_mean'].sum() if 'balance_mean' in account_summary else np.nan,
    total_loans=loan_summary['loan_amount_total'].sum() if len(loan_summary) else 0,
    total_cards=len(cards),
    total_orders=len(orders)
)

# Merge high-level stats into single row per client
client_final = client_full[['client_id', 'birth_number', 'district_id', 'A2', 'A3', 'A10', 'A11']].copy()
client_final['num_accounts'] = len(account_summary)
client_final['total_balance_mean'] = account_summary['balance_mean'].mean()
client_final['total_incoming_sum'] = account_summary['total_incoming'].sum()
client_final['total_outgoing_sum'] = account_summary['total_outgoing'].sum()
client_final['loan_amount_total'] = account_summary['loan_amount_total'].sum()
client_final['num_cards'] = len(cards)
client_final['num_orders'] = len(orders)

print(client_final)


In [ ]:
client_final

In [ ]:
import pandas as pd
import numpy as np
import glob
import os

# === Load all data ===
data_path = "/kaggle/input/masterclass"
data = {}
for path in glob.glob(os.path.join(data_path, "*.asc")):
    name = os.path.basename(path).split(".")[0]
    df = pd.read_csv(path, sep=';', quotechar='"', dtype=str, low_memory=False)
    df.columns = df.columns.str.strip()
    data[name] = df

# --- Numeric & Date parsing ---
def to_numeric(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')
    return df

data['trans'] = to_numeric(data['trans'], ['amount', 'balance'])
data['loan'] = to_numeric(data['loan'], ['amount', 'duration', 'payments'])
data['order'] = to_numeric(data['order'], ['amount'])
data['card']['issued'] = pd.to_datetime(data['card']['issued'], errors='coerce', format='%y%m%d %H:%M:%S')
data['account']['date'] = pd.to_datetime(data['account']['date'], errors='coerce', format='%y%m%d')
data['trans']['date'] = pd.to_datetime(data['trans']['date'], errors='coerce', format='%y%m%d')

# === Prepare helper aggregations ===

# --- Transaction summary per account ---
trans = data['trans'].copy()
if len(trans):
    trans_summary = trans.groupby('account_id', dropna=False).agg(
        total_transactions=('trans_id', 'count'),
        num_incoming=('type', lambda x: (x == 'PRIJEM').sum()),
        num_outgoing=('type', lambda x: (x == 'VYDAJ').sum()),
        total_incoming=('amount', lambda x: x[trans.loc[x.index, 'type'] == 'PRIJEM'].sum()),
        total_outgoing=('amount', lambda x: x[trans.loc[x.index, 'type'] == 'VYDAJ'].sum()),
        avg_incoming=('amount', lambda x: x[trans.loc[x.index, 'type'] == 'PRIJEM'].mean()),
        avg_outgoing=('amount', lambda x: x[trans.loc[x.index, 'type'] == 'VYDAJ'].mean()),
        balance_min=('balance', 'min'),
        balance_max=('balance', 'max'),
        balance_mean=('balance', 'mean'),
        unique_k_symbols=('k_symbol', 'nunique'),
        transaction_span_days=('date', lambda x: (x.max() - x.min()).days if pd.notna(x.max()) else np.nan)
    ).reset_index()
else:
    trans_summary = pd.DataFrame(columns=['account_id'])

# --- Loan summary per account ---
loan_summary = data['loan'].groupby('account_id', dropna=False).agg({
    'amount': 'sum',
    'duration': 'mean',
    'payments': 'mean',
    'status': lambda x: ','.join(x.unique())
}).rename(columns={
    'amount': 'loan_amount_total',
    'duration': 'loan_avg_duration',
    'payments': 'loan_avg_payment',
    'status': 'loan_status'
}).reset_index()

# --- Order summary per account ---
order_summary = data['order'].groupby('account_id', dropna=False).agg(
    num_orders=('amount', 'count'),
    avg_order_amount=('amount', 'mean'),
    total_order_amount=('amount', 'sum')
).reset_index()

# --- Card summary per client (via disp) ---
card_disp = data['card'].merge(data['disp'], on='disp_id', how='left')
card_summary = card_disp.groupby('client_id', dropna=False).agg(
    num_cards=('card_id', 'count'),
    earliest_card_issue=('issued', 'min')
).reset_index()

# === Account + Disp + Client joins ===
disp = data['disp'].copy()
accounts = data['account'].copy()

# Merge each account with its aggregated info
account_summary = (
    accounts
    .merge(trans_summary, on='account_id', how='left')
    .merge(loan_summary, on='account_id', how='left')
    .merge(order_summary, on='account_id', how='left')
)

# Merge disp to link clients to accounts
client_accounts = disp.merge(account_summary, on='account_id', how='left')

# Keep only OWNER relationships (each client may have multiple)
client_accounts = client_accounts[client_accounts['type'] == 'OWNER']

# === Aggregate per client ===
client_features = client_accounts.groupby('client_id', dropna=False).agg(
    num_accounts=('account_id', 'nunique'),
    total_balance_mean=('balance_mean', 'mean'),
    total_incoming_sum=('total_incoming', 'sum'),
    total_outgoing_sum=('total_outgoing', 'sum'),
    loan_amount_total=('loan_amount_total', 'sum'),
    loan_status_all=('loan_status', lambda x: ','.join(x.dropna().unique())),
    num_orders=('num_orders', 'sum')
).reset_index()

# Merge card info
client_features = client_features.merge(card_summary, on='client_id', how='left')

# Merge with client + district info
client_final = data['client'].merge(client_features, on='client_id', how='left')
client_final = client_final.merge(data['district'], left_on='district_id', right_on='A1', how='left')

# === Final dataset ===
final_cols = [
    'client_id', 'birth_number', 'district_id', 'A2', 'A3', 'A10', 'A11',
    'num_accounts', 'total_balance_mean', 'total_incoming_sum', 'total_outgoing_sum',
    'loan_amount_total', 'loan_status_all', 'num_orders', 'num_cards', 'earliest_card_issue'
]
final_dataset = client_final[final_cols].drop_duplicates().reset_index(drop=True)

print("Final dataset shape:", final_dataset.shape)
print(final_dataset.head(10))


In [ ]:
final_dataset

In [ ]:
final_dataset.shape

In [ ]:
import pandas as pd
import numpy as np
import glob
import os

# === Load all data ===
data_path = "/kaggle/input/masterclass"
data = {}
for path in glob.glob(os.path.join(data_path, "*.asc")):
    name = os.path.basename(path).split(".")[0]
    df = pd.read_csv(path, sep=';', quotechar='"', dtype=str, low_memory=False)
    df.columns = df.columns.str.strip()
    data[name] = df

# --- Numeric & Date parsing ---
def to_numeric(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')
    return df

data['trans'] = to_numeric(data['trans'], ['amount', 'balance'])
data['loan'] = to_numeric(data['loan'], ['amount', 'duration', 'payments'])
data['order'] = to_numeric(data['order'], ['amount'])
data['card']['issued'] = pd.to_datetime(data['card']['issued'], errors='coerce', format='%y%m%d %H:%M:%S')
data['account']['date'] = pd.to_datetime(data['account']['date'], errors='coerce', format='%y%m%d')
data['trans']['date'] = pd.to_datetime(data['trans']['date'], errors='coerce', format='%y%m%d')
data['loan']['date'] = pd.to_datetime(data['loan']['date'], errors='coerce', format='%y%m%d')

# === ENHANCED TRANSACTION FEATURES (25+ features for KYC, Fraud Detection, Loan Recommendation) ===
trans = data['trans'].copy()

if len(trans):
    # Filter accounts with at least one transaction
    accounts_with_trans = trans['account_id'].unique()
    
    # Basic transaction counts
    trans_summary = trans.groupby('account_id', dropna=False).agg(
        total_transactions=('trans_id', 'count'),
        num_incoming=('type', lambda x: (x == 'PRIJEM').sum()),
        num_outgoing=('type', lambda x: (x == 'VYDAJ').sum()),
        
        # Amount features
        total_incoming=('amount', lambda x: x[trans.loc[x.index, 'type'] == 'PRIJEM'].sum()),
        total_outgoing=('amount', lambda x: x[trans.loc[x.index, 'type'] == 'VYDAJ'].sum()),
        avg_incoming=('amount', lambda x: x[trans.loc[x.index, 'type'] == 'PRIJEM'].mean()),
        avg_outgoing=('amount', lambda x: x[trans.loc[x.index, 'type'] == 'VYDAJ'].mean()),
        median_incoming=('amount', lambda x: x[trans.loc[x.index, 'type'] == 'PRIJEM'].median()),
        median_outgoing=('amount', lambda x: x[trans.loc[x.index, 'type'] == 'VYDAJ'].median()),
        std_incoming=('amount', lambda x: x[trans.loc[x.index, 'type'] == 'PRIJEM'].std()),
        std_outgoing=('amount', lambda x: x[trans.loc[x.index, 'type'] == 'VYDAJ'].std()),
        max_incoming=('amount', lambda x: x[trans.loc[x.index, 'type'] == 'PRIJEM'].max()),
        max_outgoing=('amount', lambda x: x[trans.loc[x.index, 'type'] == 'VYDAJ'].max()),
        min_incoming=('amount', lambda x: x[trans.loc[x.index, 'type'] == 'PRIJEM'].min()),
        min_outgoing=('amount', lambda x: x[trans.loc[x.index, 'type'] == 'VYDAJ'].min()),
        
        # Balance features (KYC & Loan Recommendation)
        balance_min=('balance', 'min'),
        balance_max=('balance', 'max'),
        balance_mean=('balance', 'mean'),
        balance_median=('balance', 'median'),
        balance_std=('balance', 'std'),
        
        # Transaction patterns (Fraud Detection)
        unique_k_symbols=('k_symbol', 'nunique'),
        unique_operations=('operation', 'nunique'),
        unique_banks=('bank', 'nunique'),
        
        # Temporal features
        transaction_span_days=('date', lambda x: (x.max() - x.min()).days if pd.notna(x.max()) else np.nan),
        first_transaction_date=('date', 'min'),
        last_transaction_date=('date', 'max'),
    ).reset_index()
    
    # Calculate additional derived features
    trans_summary['net_cashflow'] = trans_summary['total_incoming'] - trans_summary['total_outgoing']
    trans_summary['incoming_outgoing_ratio'] = trans_summary['total_incoming'] / (trans_summary['total_outgoing'] + 1)
    trans_summary['avg_transaction_amount'] = (trans_summary['total_incoming'] + trans_summary['total_outgoing']) / trans_summary['total_transactions']
    trans_summary['transaction_frequency'] = trans_summary['total_transactions'] / (trans_summary['transaction_span_days'] + 1)
    trans_summary['balance_volatility'] = trans_summary['balance_std'] / (trans_summary['balance_mean'] + 1)
    trans_summary['incoming_volatility'] = trans_summary['std_incoming'] / (trans_summary['avg_incoming'] + 1)
    trans_summary['outgoing_volatility'] = trans_summary['std_outgoing'] / (trans_summary['avg_outgoing'] + 1)
    trans_summary['max_to_avg_incoming_ratio'] = trans_summary['max_incoming'] / (trans_summary['avg_incoming'] + 1)
    trans_summary['max_to_avg_outgoing_ratio'] = trans_summary['max_outgoing'] / (trans_summary['avg_outgoing'] + 1)
    
else:
    trans_summary = pd.DataFrame(columns=['account_id'])
    accounts_with_trans = np.array([])

# === LOAN FEATURES (Keep all original + aggregated) ===
loan = data['loan'].copy()
loan_summary = loan.groupby('account_id', dropna=False).agg({
    'loan_id': 'count',
    'amount': ['sum', 'mean', 'max', 'min'],
    'duration': ['mean', 'max', 'min'],
    'payments': ['mean', 'max', 'min'],
    'status': lambda x: ','.join(x.unique()),
    'date': ['min', 'max']
}).reset_index()

loan_summary.columns = ['account_id', 'num_loans', 'loan_amount_total', 'loan_amount_avg', 
                        'loan_amount_max', 'loan_amount_min', 'loan_duration_avg', 
                        'loan_duration_max', 'loan_duration_min', 'loan_payment_avg', 
                        'loan_payment_max', 'loan_payment_min', 'loan_status_all',
                        'first_loan_date', 'last_loan_date']

# === ORDER FEATURES (Keep all original + aggregated) ===
order = data['order'].copy()
order_summary = order.groupby('account_id', dropna=False).agg(
    num_orders=('order_id', 'count'),
    avg_order_amount=('amount', 'mean'),
    total_order_amount=('amount', 'sum'),
    max_order_amount=('amount', 'max'),
    min_order_amount=('amount', 'min'),
    std_order_amount=('amount', 'std'),
    unique_banks_orders=('bank_to', 'nunique'),
    unique_k_symbols_orders=('k_symbol', 'nunique')
).reset_index()

# === CARD FEATURES (via disp) ===
card = data['card'].copy()
disp = data['disp'].copy()

# Keep all original card features
# Note: both card and disp have 'type' column, so we need to specify suffixes
card_disp = card.merge(disp, on='disp_id', how='left', suffixes=('_card', '_disp'))
card_summary = card_disp.groupby('client_id', dropna=False).agg(
    num_cards=('card_id', 'count'),
    earliest_card_issue=('issued', 'min'),
    latest_card_issue=('issued', 'max'),
    num_classic_cards=('type_card', lambda x: (x == 'classic').sum()),
    num_junior_cards=('type_card', lambda x: (x == 'junior').sum()),
    num_gold_cards=('type_card', lambda x: (x == 'gold').sum())
).reset_index()

# === ACCOUNT + DISP + CLIENT JOINS (Keep all original features) ===
accounts = data['account'].copy()

# Merge account with all aggregated info - ONLY for accounts with transactions
account_summary = (
    accounts[accounts['account_id'].isin(accounts_with_trans)]
    .merge(trans_summary, on='account_id', how='left')
    .merge(loan_summary, on='account_id', how='left')
    .merge(order_summary, on='account_id', how='left')
)

# Merge disp to link clients to accounts
client_accounts = disp.merge(account_summary, on='account_id', how='left')

# Keep only OWNER relationships (type column from disp is just 'type')
client_accounts = client_accounts[client_accounts['type'] == 'OWNER']

# === AGGREGATE PER CLIENT (Keep all features) ===
# Numeric aggregations
numeric_agg = {
    'account_id': 'nunique',
    'total_transactions': 'sum',
    'num_incoming': 'sum',
    'num_outgoing': 'sum',
    'total_incoming': 'sum',
    'total_outgoing': 'sum',
    'avg_incoming': 'mean',
    'avg_outgoing': 'mean',
    'median_incoming': 'mean',
    'median_outgoing': 'mean',
    'std_incoming': 'mean',
    'std_outgoing': 'mean',
    'max_incoming': 'max',
    'max_outgoing': 'max',
    'min_incoming': 'min',
    'min_outgoing': 'min',
    'balance_min': 'min',
    'balance_max': 'max',
    'balance_mean': 'mean',
    'balance_median': 'mean',
    'balance_std': 'mean',
    'unique_k_symbols': 'sum',
    'unique_operations': 'sum',
    'unique_banks': 'sum',
    'transaction_span_days': 'max',
    'net_cashflow': 'sum',
    'incoming_outgoing_ratio': 'mean',
    'avg_transaction_amount': 'mean',
    'transaction_frequency': 'mean',
    'balance_volatility': 'mean',
    'incoming_volatility': 'mean',
    'outgoing_volatility': 'mean',
    'max_to_avg_incoming_ratio': 'mean',
    'max_to_avg_outgoing_ratio': 'mean',
    'num_loans': 'sum',
    'loan_amount_total': 'sum',
    'loan_amount_avg': 'mean',
    'loan_amount_max': 'max',
    'loan_amount_min': 'min',
    'loan_duration_avg': 'mean',
    'loan_duration_max': 'max',
    'loan_duration_min': 'min',
    'loan_payment_avg': 'mean',
    'loan_payment_max': 'max',
    'loan_payment_min': 'min',
    'num_orders': 'sum',
    'avg_order_amount': 'mean',
    'total_order_amount': 'sum',
    'max_order_amount': 'max',
    'min_order_amount': 'min',
    'std_order_amount': 'mean',
    'unique_banks_orders': 'sum',
    'unique_k_symbols_orders': 'sum'
}

client_features = client_accounts.groupby('client_id', dropna=False).agg(numeric_agg).reset_index()
client_features.columns = ['client_id', 'num_accounts'] + [col for col in client_features.columns[2:]]

# String aggregations
string_features = client_accounts.groupby('client_id', dropna=False).agg({
    'loan_status_all': lambda x: ','.join(x.dropna().unique()),
    'frequency': lambda x: ','.join(x.dropna().unique()),
    'first_transaction_date': 'min',
    'last_transaction_date': 'max',
    'first_loan_date': 'min',
    'last_loan_date': 'max'
}).reset_index()

client_features = client_features.merge(string_features, on='client_id', how='left')

# Merge card info
client_features = client_features.merge(card_summary, on='client_id', how='left')

# === MERGE WITH CLIENT AND DISTRICT (Keep ALL original features) ===
client_base = data['client'].copy()
district = data['district'].copy()

# Merge client with all features
client_final = client_base.merge(client_features, on='client_id', how='inner')  # inner to keep only clients with transactions

# Merge with district info (keep all district columns)
client_final = client_final.merge(district, left_on='district_id', right_on='A1', how='left')

# === FINAL DATASET ===
print("Final dataset shape:", client_final.shape)
print("\nColumn names:")
print(client_final.columns.tolist())
print("\nFirst 10 rows:")
print(client_final.head(10))
print("\nDataset info:")
print(client_final.info())
print("\nSummary statistics:")
print(client_final.describe())

# Save to CSV
client_final.to_csv('enhanced_client_features.csv', index=False)
print("\nDataset saved to 'enhanced_client_features.csv'")

In [ ]:
client_final.shape

In [ ]:
client_final.sample(5)

In [ ]:
nan_columns = client_final.columns[client_final.isna().any()]
print("Columns with NaN values:", nan_columns.tolist())


In [ ]:
nan_count_per_column = client_final.isna().sum()
nan_count_per_column = nan_count_per_column[nan_count_per_column > 0]
print(nan_count_per_column)

In [ ]:
client_final.columns

In [ ]:
client_final = client_final.fillna(0)

In [ ]:
nan_count_per_column = client_final.isna().sum()
nan_count_per_column = nan_count_per_column[nan_count_per_column > 0]
print(nan_count_per_column)

In [ ]:
client_final.to_csv("lord_finale.csv", index=True)


In [ ]:
import pandas as pd
import numpy as np
import glob
import os

# === Load all data ===
data_path = "/kaggle/input/masterclass"
data = {}
for path in glob.glob(os.path.join(data_path, "*.asc")):
    name = os.path.basename(path).split(".")[0]
    df = pd.read_csv(path, sep=';', quotechar='"', dtype=str, low_memory=False)
    df.columns = df.columns.str.strip()
    data[name] = df

# --- Numeric & Date parsing ---
def to_numeric(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')
    return df

data['trans'] = to_numeric(data['trans'], ['amount', 'balance'])
data['loan'] = to_numeric(data['loan'], ['amount', 'duration', 'payments'])
data['order'] = to_numeric(data['order'], ['amount'])

data['card']['issued'] = pd.to_datetime(data['card']['issued'], errors='coerce', format='%y%m%d %H:%M:%S')
data['account']['date'] = pd.to_datetime(data['account']['date'], errors='coerce', format='%y%m%d')
data['trans']['date'] = pd.to_datetime(data['trans']['date'], errors='coerce', format='%y%m%d')
data['loan']['date'] = pd.to_datetime(data['loan']['date'], errors='coerce', format='%y%m%d')

# === TRANSACTION FEATURES ===
trans = data['trans'].copy()

if len(trans):
    accounts_with_trans = trans['account_id'].unique()
    trans_summary = trans.groupby('account_id', dropna=False).agg(
        total_transactions=('trans_id', 'count'),
        num_incoming=('type', lambda x: (x == 'PRIJEM').sum()),
        num_outgoing=('type', lambda x: (x == 'VYDAJ').sum()),
        total_incoming=('amount', lambda x: x[trans.loc[x.index, 'type'] == 'PRIJEM'].sum()),
        total_outgoing=('amount', lambda x: x[trans.loc[x.index, 'type'] == 'VYDAJ'].sum()),
        avg_incoming=('amount', lambda x: x[trans.loc[x.index, 'type'] == 'PRIJEM'].mean()),
        avg_outgoing=('amount', lambda x: x[trans.loc[x.index, 'type'] == 'VYDAJ'].mean()),
        median_incoming=('amount', lambda x: x[trans.loc[x.index, 'type'] == 'PRIJEM'].median()),
        median_outgoing=('amount', lambda x: x[trans.loc[x.index, 'type'] == 'VYDAJ'].median()),
        std_incoming=('amount', lambda x: x[trans.loc[x.index, 'type'] == 'PRIJEM'].std()),
        std_outgoing=('amount', lambda x: x[trans.loc[x.index, 'type'] == 'VYDAJ'].std()),
        max_incoming=('amount', lambda x: x[trans.loc[x.index, 'type'] == 'PRIJEM'].max()),
        max_outgoing=('amount', lambda x: x[trans.loc[x.index, 'type'] == 'VYDAJ'].max()),
        min_incoming=('amount', lambda x: x[trans.loc[x.index, 'type'] == 'PRIJEM'].min()),
        min_outgoing=('amount', lambda x: x[trans.loc[x.index, 'type'] == 'VYDAJ'].min()),
        balance_min=('balance', 'min'),
        balance_max=('balance', 'max'),
        balance_mean=('balance', 'mean'),
        balance_median=('balance', 'median'),
        balance_std=('balance', 'std'),
        unique_k_symbols=('k_symbol', 'nunique'),
        unique_operations=('operation', 'nunique'),
        unique_banks=('bank', 'nunique'),
        transaction_span_days=('date', lambda x: (x.max() - x.min()).days if pd.notna(x.max()) else np.nan),
        first_transaction_date=('date', 'min'),
        last_transaction_date=('date', 'max'),
    ).reset_index()

    # Derived features
    trans_summary['net_cashflow'] = trans_summary['total_incoming'] - trans_summary['total_outgoing']
    trans_summary['incoming_outgoing_ratio'] = trans_summary['total_incoming'] / (trans_summary['total_outgoing'] + 1)
    trans_summary['avg_transaction_amount'] = (trans_summary['total_incoming'] + trans_summary['total_outgoing']) / trans_summary['total_transactions']
    trans_summary['transaction_frequency'] = trans_summary['total_transactions'] / (trans_summary['transaction_span_days'] + 1)
    trans_summary['balance_volatility'] = trans_summary['balance_std'] / (trans_summary['balance_mean'] + 1)
    trans_summary['incoming_volatility'] = trans_summary['std_incoming'] / (trans_summary['avg_incoming'] + 1)
    trans_summary['outgoing_volatility'] = trans_summary['std_outgoing'] / (trans_summary['avg_outgoing'] + 1)
    trans_summary['max_to_avg_incoming_ratio'] = trans_summary['max_incoming'] / (trans_summary['avg_incoming'] + 1)
    trans_summary['max_to_avg_outgoing_ratio'] = trans_summary['max_outgoing'] / (trans_summary['avg_outgoing'] + 1)
else:
    trans_summary = pd.DataFrame(columns=['account_id'])
    accounts_with_trans = np.array([])

# === LOAN FEATURES ===
loan = data['loan'].copy()
loan_summary = loan.groupby('account_id', dropna=False).agg({
    'loan_id': 'count',
    'amount': ['sum', 'mean', 'max', 'min'],
    'duration': ['mean', 'max', 'min'],
    'payments': ['mean', 'max', 'min'],
    'status': lambda x: ','.join(x.unique()),
    'date': ['min', 'max']
}).reset_index()

loan_summary.columns = ['account_id', 'num_loans', 'loan_amount_total', 'loan_amount_avg',
                        'loan_amount_max', 'loan_amount_min', 'loan_duration_avg',
                        'loan_duration_max', 'loan_duration_min', 'loan_payment_avg',
                        'loan_payment_max', 'loan_payment_min', 'loan_status_all',
                        'first_loan_date', 'last_loan_date']

# === ORDER FEATURES ===
order = data['order'].copy()
order_summary = order.groupby('account_id', dropna=False).agg(
    num_orders=('order_id', 'count'),
    avg_order_amount=('amount', 'mean'),
    total_order_amount=('amount', 'sum'),
    max_order_amount=('amount', 'max'),
    min_order_amount=('amount', 'min'),
    std_order_amount=('amount', 'std'),
    unique_banks_orders=('bank_to', 'nunique'),
    unique_k_symbols_orders=('k_symbol', 'nunique')
).reset_index()

# === ACCOUNT + DISP + CLIENT JOINS (KEEP ALL IDs) ===
accounts = data['account'].copy()
disp = data['disp'].copy()

account_summary = (
    accounts.merge(trans_summary, on='account_id', how='left')
            .merge(loan_summary, on='account_id', how='left')
            .merge(order_summary, on='account_id', how='left')
)

client_accounts = disp.merge(account_summary, on='account_id', how='left')

# Merge card
card_disp = data['card'].merge(disp, on='disp_id', how='left')
client_accounts = client_accounts.merge(
    card_disp[['disp_id', 'client_id', 'account_id', 'card_id', 'issued']],
    on=['client_id', 'account_id'], how='left'
)

# Add sample transaction/loan/order IDs
for name in ['trans', 'loan', 'order']:
    df = data[name][['account_id', f'{name}_id']].drop_duplicates('account_id')
    client_accounts = client_accounts.merge(df, on='account_id', how='left')

# === AGGREGATION (NUMERIC + STRING + ID LISTS) ===
numeric_agg = {
    'account_id': 'nunique',
    'total_transactions': 'sum',
    'num_incoming': 'sum',
    'num_outgoing': 'sum',
    'total_incoming': 'sum',
    'total_outgoing': 'sum',
    'avg_incoming': 'mean',
    'avg_outgoing': 'mean',
    'median_incoming': 'mean',
    'median_outgoing': 'mean',
    'std_incoming': 'mean',
    'std_outgoing': 'mean',
    'max_incoming': 'max',
    'max_outgoing': 'max',
    'min_incoming': 'min',
    'min_outgoing': 'min',
    'balance_min': 'min',
    'balance_max': 'max',
    'balance_mean': 'mean',
    'balance_median': 'mean',
    'balance_std': 'mean',
    'unique_k_symbols': 'sum',
    'unique_operations': 'sum',
    'unique_banks': 'sum',
    'transaction_span_days': 'max',
    'net_cashflow': 'sum',
    'incoming_outgoing_ratio': 'mean',
    'avg_transaction_amount': 'mean',
    'transaction_frequency': 'mean',
    'balance_volatility': 'mean',
    'incoming_volatility': 'mean',
    'outgoing_volatility': 'mean',
    'max_to_avg_incoming_ratio': 'mean',
    'max_to_avg_outgoing_ratio': 'mean',
    'num_loans': 'sum',
    'loan_amount_total': 'sum',
    'loan_amount_avg': 'mean',
    'loan_amount_max': 'max',
    'loan_amount_min': 'min',
    'loan_duration_avg': 'mean',
    'loan_duration_max': 'max',
    'loan_duration_min': 'min',
    'loan_payment_avg': 'mean',
    'loan_payment_max': 'max',
    'loan_payment_min': 'min',
    'num_orders': 'sum',
    'avg_order_amount': 'mean',
    'total_order_amount': 'sum',
    'max_order_amount': 'max',
    'min_order_amount': 'min',
    'std_order_amount': 'mean',
    'unique_banks_orders': 'sum',
    'unique_k_symbols_orders': 'sum'
}

client_features = client_accounts.groupby('client_id', dropna=False).agg(numeric_agg).reset_index()
client_features.columns = ['client_id', 'num_accounts'] + [col for col in client_features.columns[2:]]

string_features = client_accounts.groupby('client_id', dropna=False).agg({
    'loan_status_all': lambda x: ','.join(x.dropna().unique()),
    'frequency': lambda x: ','.join(x.dropna().unique()),
    'first_transaction_date': 'min',
    'last_transaction_date': 'max',
    'first_loan_date': 'min',
    'last_loan_date': 'max'
}).reset_index()

# === FIXED ID FEATURES ===
cols_present = [c for c in ['account_id', 'disp_id', 'card_id', 'loan_id', 'order_id', 'trans_id'] if c in client_accounts.columns]
id_features = client_accounts.groupby('client_id', dropna=False).agg({
    c: lambda x: ','.join(x.dropna().astype(str).unique()) for c in cols_present
}).reset_index()

# === FINAL CLIENT MERGE ===
client_final = (
    data['client']
    .merge(client_features, on='client_id', how='left')
    .merge(string_features, on='client_id', how='left')
    .merge(id_features, on='client_id', how='left')
    .merge(data['district'], left_on='district_id', right_on='A1', how='left')
)

# === FINAL OUTPUT ===
print("✅ Final dataset shape:", client_final.shape)
print("\n🧩 Columns in final dataset:")
print(client_final.columns.tolist())
print("\n🔍 Preview:")
print(client_final.head(10))
print("\nℹ️ Dataset info:")
print(client_final.info())

client_final.to_csv('enhanced_client_features_with_ids.csv', index=False)
print("\n💾 Saved to 'enhanced_client_features_with_ids.csv'")


In [ ]:
import pandas as pd
import numpy as np
import glob
import os

# === Load all data ===
data_path = "/kaggle/input/masterclass"
data = {}
for path in glob.glob(os.path.join(data_path, "*.asc")):
    name = os.path.basename(path).split(".")[0]
    df = pd.read_csv(path, sep=';', quotechar='"', dtype=str, low_memory=False)
    df.columns = df.columns.str.strip()
    data[name] = df

# --- Numeric & Date parsing ---
def to_numeric(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')
    return df

data['trans'] = to_numeric(data['trans'], ['amount', 'balance'])
data['loan'] = to_numeric(data['loan'], ['amount', 'duration', 'payments'])
data['order'] = to_numeric(data['order'], ['amount'])

# Parse dates
data['card']['issued'] = pd.to_datetime(data['card']['issued'], errors='coerce', format='%y%m%d %H:%M:%S')
data['account']['date'] = pd.to_datetime(data['account']['date'], errors='coerce', format='%y%m%d')
data['trans']['date'] = pd.to_datetime(data['trans']['date'], errors='coerce', format='%y%m%d')
data['loan']['date'] = pd.to_datetime(data['loan']['date'], errors='coerce', format='%y%m%d')

# === TRANSACTION FEATURES ===
trans = data['trans'].copy()
if len(trans):
    accounts_with_trans = trans['account_id'].unique()
    trans_summary = trans.groupby('account_id', dropna=False).agg(
        total_transactions=('trans_id', 'count'),
        num_incoming=('type', lambda x: (x == 'PRIJEM').sum()),
        num_outgoing=('type', lambda x: (x == 'VYDAJ').sum()),
        total_incoming=('amount', lambda x: x[trans.loc[x.index, 'type'] == 'PRIJEM'].sum()),
        total_outgoing=('amount', lambda x: x[trans.loc[x.index, 'type'] == 'VYDAJ'].sum()),
        avg_incoming=('amount', lambda x: x[trans.loc[x.index, 'type'] == 'PRIJEM'].mean()),
        avg_outgoing=('amount', lambda x: x[trans.loc[x.index, 'type'] == 'VYDAJ'].mean()),
        median_incoming=('amount', lambda x: x[trans.loc[x.index, 'type'] == 'PRIJEM'].median()),
        median_outgoing=('amount', lambda x: x[trans.loc[x.index, 'type'] == 'VYDAJ'].median()),
        std_incoming=('amount', lambda x: x[trans.loc[x.index, 'type'] == 'PRIJEM'].std()),
        std_outgoing=('amount', lambda x: x[trans.loc[x.index, 'type'] == 'VYDAJ'].std()),
        max_incoming=('amount', lambda x: x[trans.loc[x.index, 'type'] == 'PRIJEM'].max()),
        max_outgoing=('amount', lambda x: x[trans.loc[x.index, 'type'] == 'VYDAJ'].max()),
        min_incoming=('amount', lambda x: x[trans.loc[x.index, 'type'] == 'PRIJEM'].min()),
        min_outgoing=('amount', lambda x: x[trans.loc[x.index, 'type'] == 'VYDAJ'].min()),
        balance_min=('balance', 'min'),
        balance_max=('balance', 'max'),
        balance_mean=('balance', 'mean'),
        balance_median=('balance', 'median'),
        balance_std=('balance', 'std'),
        unique_k_symbols=('k_symbol', 'nunique'),
        unique_operations=('operation', 'nunique'),
        unique_banks=('bank', 'nunique'),
        transaction_span_days=('date', lambda x: (x.max() - x.min()).days if pd.notna(x.max()) else np.nan),
        first_transaction_date=('date', 'min'),
        last_transaction_date=('date', 'max'),
    ).reset_index()

    # Derived metrics
    trans_summary['net_cashflow'] = trans_summary['total_incoming'] - trans_summary['total_outgoing']
    trans_summary['incoming_outgoing_ratio'] = trans_summary['total_incoming'] / (trans_summary['total_outgoing'] + 1)
    trans_summary['avg_transaction_amount'] = (trans_summary['total_incoming'] + trans_summary['total_outgoing']) / trans_summary['total_transactions']
    trans_summary['transaction_frequency'] = trans_summary['total_transactions'] / (trans_summary['transaction_span_days'] + 1)
    trans_summary['balance_volatility'] = trans_summary['balance_std'] / (trans_summary['balance_mean'] + 1)
    trans_summary['incoming_volatility'] = trans_summary['std_incoming'] / (trans_summary['avg_incoming'] + 1)
    trans_summary['outgoing_volatility'] = trans_summary['std_outgoing'] / (trans_summary['avg_outgoing'] + 1)
    trans_summary['max_to_avg_incoming_ratio'] = trans_summary['max_incoming'] / (trans_summary['avg_incoming'] + 1)
    trans_summary['max_to_avg_outgoing_ratio'] = trans_summary['max_outgoing'] / (trans_summary['avg_outgoing'] + 1)
else:
    trans_summary = pd.DataFrame(columns=['account_id'])
    accounts_with_trans = np.array([])

# === LOAN FEATURES ===
loan = data['loan'].copy()
loan_summary = loan.groupby('account_id', dropna=False).agg({
    'loan_id': 'count',
    'amount': ['sum', 'mean', 'max', 'min'],
    'duration': ['mean', 'max', 'min'],
    'payments': ['mean', 'max', 'min'],
    'status': lambda x: ','.join(x.unique()),
    'date': ['min', 'max']
}).reset_index()

loan_summary.columns = ['account_id', 'num_loans', 'loan_amount_total', 'loan_amount_avg',
                        'loan_amount_max', 'loan_amount_min', 'loan_duration_avg',
                        'loan_duration_max', 'loan_duration_min', 'loan_payment_avg',
                        'loan_payment_max', 'loan_payment_min', 'loan_status_all',
                        'first_loan_date', 'last_loan_date']

# === ORDER FEATURES ===
order = data['order'].copy()
order_summary = order.groupby('account_id', dropna=False).agg(
    num_orders=('order_id', 'count'),
    avg_order_amount=('amount', 'mean'),
    total_order_amount=('amount', 'sum'),
    max_order_amount=('amount', 'max'),
    min_order_amount=('amount', 'min'),
    std_order_amount=('amount', 'std'),
    unique_banks_orders=('bank_to', 'nunique'),
    unique_k_symbols_orders=('k_symbol', 'nunique')
).reset_index()

# === CARD FEATURES ===
card = data['card'].copy()
disp = data['disp'].copy()

card_disp = card.merge(disp, on='disp_id', how='left', suffixes=('_card', '_disp'))
card_summary = card_disp.groupby('client_id', dropna=False).agg(
    num_cards=('card_id', 'count'),
    earliest_card_issue=('issued', 'min'),
    latest_card_issue=('issued', 'max'),
    num_classic_cards=('type_card', lambda x: (x == 'classic').sum()),
    num_junior_cards=('type_card', lambda x: (x == 'junior').sum()),
    num_gold_cards=('type_card', lambda x: (x == 'gold').sum())
).reset_index()

# === ACCOUNT + DISP + CLIENT JOINS (with all IDs) ===
accounts = data['account'].copy()

account_summary = (
    accounts.merge(trans_summary, on='account_id', how='left')
            .merge(loan_summary, on='account_id', how='left')
            .merge(order_summary, on='account_id', how='left')
)

client_accounts = disp.merge(account_summary, on='account_id', how='left')

# Merge card IDs
card_disp_ids = data['card'].merge(disp, on='disp_id', how='left')
client_accounts = client_accounts.merge(
    card_disp_ids[['disp_id', 'client_id', 'account_id', 'card_id', 'issued']],
    on=['client_id', 'account_id'], how='left'
)

# Add sample transaction/loan/order IDs
for name in ['trans', 'loan', 'order']:
    df = data[name][['account_id', f'{name}_id']].drop_duplicates('account_id')
    client_accounts = client_accounts.merge(df, on='account_id', how='left')

# === AGGREGATION ===
numeric_agg = {k: v for k, v in [
    ('account_id', 'nunique'),
    ('total_transactions', 'sum'),
    ('num_incoming', 'sum'),
    ('num_outgoing', 'sum'),
    ('total_incoming', 'sum'),
    ('total_outgoing', 'sum'),
    ('avg_incoming', 'mean'),
    ('avg_outgoing', 'mean'),
    ('median_incoming', 'mean'),
    ('median_outgoing', 'mean'),
    ('std_incoming', 'mean'),
    ('std_outgoing', 'mean'),
    ('max_incoming', 'max'),
    ('max_outgoing', 'max'),
    ('min_incoming', 'min'),
    ('min_outgoing', 'min'),
    ('balance_min', 'min'),
    ('balance_max', 'max'),
    ('balance_mean', 'mean'),
    ('balance_median', 'mean'),
    ('balance_std', 'mean'),
    ('unique_k_symbols', 'sum'),
    ('unique_operations', 'sum'),
    ('unique_banks', 'sum'),
    ('transaction_span_days', 'max'),
    ('net_cashflow', 'sum'),
    ('incoming_outgoing_ratio', 'mean'),
    ('avg_transaction_amount', 'mean'),
    ('transaction_frequency', 'mean'),
    ('balance_volatility', 'mean'),
    ('incoming_volatility', 'mean'),
    ('outgoing_volatility', 'mean'),
    ('max_to_avg_incoming_ratio', 'mean'),
    ('max_to_avg_outgoing_ratio', 'mean'),
    ('num_loans', 'sum'),
    ('loan_amount_total', 'sum'),
    ('loan_amount_avg', 'mean'),
    ('loan_amount_max', 'max'),
    ('loan_amount_min', 'min'),
    ('loan_duration_avg', 'mean'),
    ('loan_duration_max', 'max'),
    ('loan_duration_min', 'min'),
    ('loan_payment_avg', 'mean'),
    ('loan_payment_max', 'max'),
    ('loan_payment_min', 'min'),
    ('num_orders', 'sum'),
    ('avg_order_amount', 'mean'),
    ('total_order_amount', 'sum'),
    ('max_order_amount', 'max'),
    ('min_order_amount', 'min'),
    ('std_order_amount', 'mean'),
    ('unique_banks_orders', 'sum'),
    ('unique_k_symbols_orders', 'sum')
]}

client_features = client_accounts.groupby('client_id', dropna=False).agg(numeric_agg).reset_index()
client_features.columns = ['client_id', 'num_accounts'] + [col for col in client_features.columns[2:]]

string_features = client_accounts.groupby('client_id', dropna=False).agg({
    'loan_status_all': lambda x: ','.join(x.dropna().unique()),
    'frequency': lambda x: ','.join(x.dropna().unique()),
    'first_transaction_date': 'min',
    'last_transaction_date': 'max',
    'first_loan_date': 'min',
    'last_loan_date': 'max'
}).reset_index()

cols_present = [c for c in ['account_id', 'disp_id', 'card_id', 'loan_id', 'order_id', 'trans_id'] if c in client_accounts.columns]
id_features = client_accounts.groupby('client_id', dropna=False).agg({
    c: lambda x: ','.join(x.dropna().astype(str).unique()) for c in cols_present
}).reset_index()

# === FINAL CLIENT MERGE ===
client_final = (
    data['client']
    .merge(client_features, on='client_id', how='left')
    .merge(string_features, on='client_id', how='left')
    .merge(id_features, on='client_id', how='left')
    .merge(card_summary, on='client_id', how='left')
    .merge(data['district'], left_on='district_id', right_on='A1', how='left')
)

print("✅ Final dataset shape:", client_final.shape)
print("\n🧩 Columns in final dataset:")
print(client_final.columns.tolist())
print("\n💾 Saving...")
client_final.to_csv('enhanced_client_features_with_ids_and_cardstats.csv', index=False)
print("\n✅ Saved to 'enhanced_client_features_with_ids_and_cardstats.csv'")


In [ ]:
import pandas as pd
import numpy as np
import glob
import os

# === Load all data ===
data_path = "/kaggle/input/masterclass"
data = {}
for path in glob.glob(os.path.join(data_path, "*.asc")):
    name = os.path.basename(path).split(".")[0]
    df = pd.read_csv(path, sep=';', quotechar='"', dtype=str, low_memory=False)
    df.columns = df.columns.str.strip()
    data[name] = df

# --- Numeric & Date parsing ---
def to_numeric(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')
    return df

data['trans'] = to_numeric(data['trans'], ['amount', 'balance'])
data['loan'] = to_numeric(data['loan'], ['amount', 'duration', 'payments'])
data['order'] = to_numeric(data['order'], ['amount'])

# --- Currency Conversion to USD ---
CZK_TO_USD = 0.038

def convert_to_usd(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = df[c] * CZK_TO_USD
    return df

# Apply conversion
data['trans'] = convert_to_usd(data['trans'], ['amount', 'balance'])
data['loan'] = convert_to_usd(data['loan'], ['amount', 'payments'])
data['order'] = convert_to_usd(data['order'], ['amount'])

# --- Parse dates ---
data['card']['issued'] = pd.to_datetime(data['card']['issued'], errors='coerce', format='%y%m%d %H:%M:%S')
data['account']['date'] = pd.to_datetime(data['account']['date'], errors='coerce', format='%y%m%d')
data['trans']['date'] = pd.to_datetime(data['trans']['date'], errors='coerce', format='%y%m%d')
data['loan']['date'] = pd.to_datetime(data['loan']['date'], errors='coerce', format='%y%m%d')

# === TRANSACTION FEATURES ===
trans = data['trans'].copy()
if len(trans):
    accounts_with_trans = trans['account_id'].unique()
    trans_summary = trans.groupby('account_id', dropna=False).agg(
        total_transactions=('trans_id', 'count'),
        num_incoming=('type', lambda x: (x == 'PRIJEM').sum()),
        num_outgoing=('type', lambda x: (x == 'VYDAJ').sum()),
        total_incoming=('amount', lambda x: x[trans.loc[x.index, 'type'] == 'PRIJEM'].sum()),
        total_outgoing=('amount', lambda x: x[trans.loc[x.index, 'type'] == 'VYDAJ'].sum()),
        avg_incoming=('amount', lambda x: x[trans.loc[x.index, 'type'] == 'PRIJEM'].mean()),
        avg_outgoing=('amount', lambda x: x[trans.loc[x.index, 'type'] == 'VYDAJ'].mean()),
        median_incoming=('amount', lambda x: x[trans.loc[x.index, 'type'] == 'PRIJEM'].median()),
        median_outgoing=('amount', lambda x: x[trans.loc[x.index, 'type'] == 'VYDAJ'].median()),
        std_incoming=('amount', lambda x: x[trans.loc[x.index, 'type'] == 'PRIJEM'].std()),
        std_outgoing=('amount', lambda x: x[trans.loc[x.index, 'type'] == 'VYDAJ'].std()),
        max_incoming=('amount', lambda x: x[trans.loc[x.index, 'type'] == 'PRIJEM'].max()),
        max_outgoing=('amount', lambda x: x[trans.loc[x.index, 'type'] == 'VYDAJ'].max()),
        min_incoming=('amount', lambda x: x[trans.loc[x.index, 'type'] == 'PRIJEM'].min()),
        min_outgoing=('amount', lambda x: x[trans.loc[x.index, 'type'] == 'VYDAJ'].min()),
        balance_min=('balance', 'min'),
        balance_max=('balance', 'max'),
        balance_mean=('balance', 'mean'),
        balance_median=('balance', 'median'),
        balance_std=('balance', 'std'),
        unique_k_symbols=('k_symbol', 'nunique'),
        unique_operations=('operation', 'nunique'),
        unique_banks=('bank', 'nunique'),
        transaction_span_days=('date', lambda x: (x.max() - x.min()).days if pd.notna(x.max()) else np.nan),
        first_transaction_date=('date', 'min'),
        last_transaction_date=('date', 'max'),
    ).reset_index()

    # Derived metrics
    trans_summary['net_cashflow'] = trans_summary['total_incoming'] - trans_summary['total_outgoing']
    trans_summary['incoming_outgoing_ratio'] = trans_summary['total_incoming'] / (trans_summary['total_outgoing'] + 1)
    trans_summary['avg_transaction_amount'] = (trans_summary['total_incoming'] + trans_summary['total_outgoing']) / trans_summary['total_transactions']
    trans_summary['transaction_frequency'] = trans_summary['total_transactions'] / (trans_summary['transaction_span_days'] + 1)
    trans_summary['balance_volatility'] = trans_summary['balance_std'] / (trans_summary['balance_mean'] + 1)
    trans_summary['incoming_volatility'] = trans_summary['std_incoming'] / (trans_summary['avg_incoming'] + 1)
    trans_summary['outgoing_volatility'] = trans_summary['std_outgoing'] / (trans_summary['avg_outgoing'] + 1)
    trans_summary['max_to_avg_incoming_ratio'] = trans_summary['max_incoming'] / (trans_summary['avg_incoming'] + 1)
    trans_summary['max_to_avg_outgoing_ratio'] = trans_summary['max_outgoing'] / (trans_summary['avg_outgoing'] + 1)
else:
    trans_summary = pd.DataFrame(columns=['account_id'])
    accounts_with_trans = np.array([])

# === LOAN FEATURES ===
loan = data['loan'].copy()
loan_summary = loan.groupby('account_id', dropna=False).agg({
    'loan_id': 'count',
    'amount': ['sum', 'mean', 'max', 'min'],
    'duration': ['mean', 'max', 'min'],
    'payments': ['mean', 'max', 'min'],
    'status': lambda x: ','.join(x.unique()),
    'date': ['min', 'max']
}).reset_index()

loan_summary.columns = ['account_id', 'num_loans', 'loan_amount_total_usd', 'loan_amount_avg_usd',
                        'loan_amount_max_usd', 'loan_amount_min_usd', 'loan_duration_avg',
                        'loan_duration_max', 'loan_duration_min', 'loan_payment_avg_usd',
                        'loan_payment_max_usd', 'loan_payment_min_usd', 'loan_status_all',
                        'first_loan_date', 'last_loan_date']

# === ORDER FEATURES ===
order = data['order'].copy()
order_summary = order.groupby('account_id', dropna=False).agg(
    num_orders=('order_id', 'count'),
    avg_order_amount_usd=('amount', 'mean'),
    total_order_amount_usd=('amount', 'sum'),
    max_order_amount_usd=('amount', 'max'),
    min_order_amount_usd=('amount', 'min'),
    std_order_amount_usd=('amount', 'std'),
    unique_banks_orders=('bank_to', 'nunique'),
    unique_k_symbols_orders=('k_symbol', 'nunique')
).reset_index()

# === CARD FEATURES ===
card = data['card'].copy()
disp = data['disp'].copy()

card_disp = card.merge(disp, on='disp_id', how='left', suffixes=('_card', '_disp'))
card_summary = card_disp.groupby('client_id', dropna=False).agg(
    num_cards=('card_id', 'count'),
    earliest_card_issue=('issued', 'min'),
    latest_card_issue=('issued', 'max'),
    num_classic_cards=('type_card', lambda x: (x == 'classic').sum()),
    num_junior_cards=('type_card', lambda x: (x == 'junior').sum()),
    num_gold_cards=('type_card', lambda x: (x == 'gold').sum())
).reset_index()

# === ACCOUNT + DISP + CLIENT JOINS ===
accounts = data['account'].copy()

account_summary = (
    accounts.merge(trans_summary, on='account_id', how='left')
            .merge(loan_summary, on='account_id', how='left')
            .merge(order_summary, on='account_id', how='left')
)

client_accounts = disp.merge(account_summary, on='account_id', how='left')

card_disp_ids = data['card'].merge(disp, on='disp_id', how='left')
client_accounts = client_accounts.merge(
    card_disp_ids[['disp_id', 'client_id', 'account_id', 'card_id', 'issued']],
    on=['client_id', 'account_id'], how='left'
)

for name in ['trans', 'loan', 'order']:
    df = data[name][['account_id', f'{name}_id']].drop_duplicates('account_id')
    client_accounts = client_accounts.merge(df, on='account_id', how='left')

# === AGGREGATION ===
numeric_agg = {k: v for k, v in [
    ('account_id', 'nunique'),
    ('total_transactions', 'sum'),
    ('num_incoming', 'sum'),
    ('num_outgoing', 'sum'),
    ('total_incoming', 'sum'),
    ('total_outgoing', 'sum'),
    ('avg_incoming', 'mean'),
    ('avg_outgoing', 'mean'),
    ('median_incoming', 'mean'),
    ('median_outgoing', 'mean'),
    ('std_incoming', 'mean'),
    ('std_outgoing', 'mean'),
    ('max_incoming', 'max'),
    ('max_outgoing', 'max'),
    ('min_incoming', 'min'),
    ('min_outgoing', 'min'),
    ('balance_min', 'min'),
    ('balance_max', 'max'),
    ('balance_mean', 'mean'),
    ('balance_median', 'mean'),
    ('balance_std', 'mean'),
    ('unique_k_symbols', 'sum'),
    ('unique_operations', 'sum'),
    ('unique_banks', 'sum'),
    ('transaction_span_days', 'max'),
    ('net_cashflow', 'sum'),
    ('incoming_outgoing_ratio', 'mean'),
    ('avg_transaction_amount', 'mean'),
    ('transaction_frequency', 'mean'),
    ('balance_volatility', 'mean'),
    ('incoming_volatility', 'mean'),
    ('outgoing_volatility', 'mean'),
    ('max_to_avg_incoming_ratio', 'mean'),
    ('max_to_avg_outgoing_ratio', 'mean'),
    ('num_loans', 'sum'),
    ('loan_amount_total_usd', 'sum'),
    ('loan_amount_avg_usd', 'mean'),
    ('loan_amount_max_usd', 'max'),
    ('loan_amount_min_usd', 'min'),
    ('loan_duration_avg', 'mean'),
    ('loan_duration_max', 'max'),
    ('loan_duration_min', 'min'),
    ('loan_payment_avg_usd', 'mean'),
    ('loan_payment_max_usd', 'max'),
    ('loan_payment_min_usd', 'min'),
    ('num_orders', 'sum'),
    ('avg_order_amount_usd', 'mean'),
    ('total_order_amount_usd', 'sum'),
    ('max_order_amount_usd', 'max'),
    ('min_order_amount_usd', 'min'),
    ('std_order_amount_usd', 'mean'),
    ('unique_banks_orders', 'sum'),
    ('unique_k_symbols_orders', 'sum')
]}

client_features = client_accounts.groupby('client_id', dropna=False).agg(numeric_agg).reset_index()
client_features.columns = ['client_id', 'num_accounts'] + [col for col in client_features.columns[2:]]

string_features = client_accounts.groupby('client_id', dropna=False).agg({
    'loan_status_all': lambda x: ','.join(x.dropna().unique()),
    'frequency': lambda x: ','.join(x.dropna().unique()),
    'first_transaction_date': 'min',
    'last_transaction_date': 'max',
    'first_loan_date': 'min',
    'last_loan_date': 'max'
}).reset_index()

cols_present = [c for c in ['account_id', 'disp_id', 'card_id', 'loan_id', 'order_id', 'trans_id'] if c in client_accounts.columns]
id_features = client_accounts.groupby('client_id', dropna=False).agg({
    c: lambda x: ','.join(x.dropna().astype(str).unique()) for c in cols_present
}).reset_index()

# === FINAL CLIENT MERGE ===
client_final = (
    data['client']
    .merge(client_features, on='client_id', how='left')
    .merge(string_features, on='client_id', how='left')
    .merge(id_features, on='client_id', how='left')
    .merge(card_summary, on='client_id', how='left')
    .merge(data['district'], left_on='district_id', right_on='A1', how='left')
)

print("✅ Final dataset shape:", client_final.shape)
print("\n🧩 Columns in final dataset:")
print(client_final.columns.tolist())
print("\n💾 Saving...")
client_final.to_csv('enhanced_client_features_with_ids_and_cardstats_usd.csv', index=False)
print("\n✅ Saved to 'enhanced_client_features_with_ids_and_cardstats_usd.csv'")


In [ ]:
dff = pd.read_csv("/kaggle/input/sensational/loans_types_data.csv")
print(dff.columns)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
import pandas as pd

# Load both datasets
czech = pd.read_csv("/kaggle/input/clean-master/enhanced_client_features_with_ids_and_cardstats_usd.csv")
us = pd.read_csv("/kaggle/input/sensational/loans_types_data.csv")

# Select comparable features
czech_features = [
    'balance_mean', 'total_incoming', 'total_outgoing',
    'loan_amount_total_usd', 'loan_payment_avg_usd',
    'loan_duration_avg', 'num_loans', 'num_cards', 'net_cashflow'
]
us_features = [
    'annual_income', 'total_credit_utilized',
    'loan_amount', 'installment', 'term',
    'num_mort_accounts', 'num_total_cc_accounts',
    'debt_to_income'
]

# Clean and normalize
czech_scaled = StandardScaler().fit_transform(czech[czech_features].fillna(0))
us_scaled = StandardScaler().fit_transform(us[us_features].fillna(0))

# Find nearest US record for each Czech client
nbrs = NearestNeighbors(n_neighbors=3, metric='euclidean').fit(us_scaled)
distances, indices = nbrs.kneighbors(czech_scaled)

# Map purposes
matched_purposes = us.iloc[indices.flatten()]['loan_purpose'].values
czech['loan_purpose'] = matched_purposes
